<a href="https://colab.research.google.com/github/HongYeonLee/Artificial-Intelligence/blob/main/HW_2371049.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part A. Understanding ResNet by Hand

#### A1 Residual Forward & Backward by Hand


주어진 식: $y = F(x) + x$, 여기서 $F(x) = W_2 \cdot \text{ReLU}(W_1 \cdot x + b_1) + b_2$

<br> 1단계: $z_1 = W_1 \cdot x + b_1$ 계산

$$z_1 = \begin{bmatrix} 0.5 & -0.2 \\ -0.3 & 0.4 \end{bmatrix} \cdot \begin{bmatrix} 1.0 \\ -0.5 \end{bmatrix} + \begin{bmatrix} 0.0 \\ 0.1 \end{bmatrix}$$

- 상단 원소: $(0.5 \times 1.0) + (-0.2 \times -0.5) + 0.0 = 0.5 + 0.1 + 0.0 = 0.6$
- 하단 원소: $(-0.3 \times 1.0) + (0.4 \times -0.5) + 0.1 = -0.3 - 0.2 + 0.1 = -0.4$
- 결과: $z_1 = [0.6, -0.4]$

<br> 2단계: $h_1 = \text{ReLU}(z_1)$

- $h_1 = [\max(0, 0.6), \max(0, -0.4)] = [0.6, 0.0]$

- 결과: $h_1 = [0.6, 0.0]$

<br> 3단계: $F(x) = W_2 \cdot h_1 + b_2$

$$F(x) = \begin{bmatrix} 0.6 & 0.1 \\ 0.2 & -0.5 \end{bmatrix} \cdot \begin{bmatrix} 0.6 \\ 0.0 \end{bmatrix} + \begin{bmatrix} 0.0 \\ 0.0 \end{bmatrix}$$

- 상단: $(0.6 \times 0.6) + (0.1 \times 0.0) = 0.36$
- 하단: $(0.2 \times 0.6) + (-0.5 \times 0.0) = 0.12$
- 결과: $F(x) = [0.36, 0.12]$

<br> 4단계: $y = F(x) + x$
$$y = [0.36, 0.12] + [1.0, -0.5] = [1.36, -0.38]$$
- 결과: $y = [1.36, -0.38]$

Loss Gradient: $\frac{\partial L}{\partial y}$
<br> Loss function이 $L = 0.5 \cdot \|y - t\|^2$ 이므로 $$\frac{\partial L}{\partial y} = y - t$$
- $y = [1.36, -0.38], t = [0.3, 0.0]$
- $\frac{\partial L}{\partial y} = [1.06, -0.38]$

<br> (1) $\frac{\partial L}{\partial F}$ 유도

$$\frac{\partial L}{\partial F} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial F}$$

<br> 위 식 $y = F + x$를 $F$에 대해 미분하면 $\frac{\partial y}{\partial F} = 1$ (단위 행렬 $I$)이므로$$\frac{\partial L}{\partial F} = \frac{\partial L}{\partial y} \cdot 1 = \frac{\partial L}{\partial y}$$

<br><br> (2) $\frac{\partial L}{\partial x_{skip}}$ 유도
$$\frac{\partial L}{\partial x_{skip}} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}$$

<br> 마찬가지로 $y = F + x$를 $x$에 대해 미분하면 $\frac{\partial y}{\partial x} = 1$ (단위 행렬 $I$)이므로 <br><br> $$\frac{\partial L}{\partial x_{skip}} = \frac{\partial L}{\partial y} \cdot 1 = \frac{\partial L}{\partial y}$$

따라서
<br>
$$\frac{\partial y}{\partial x} = \frac{\partial}{\partial x}(F(x) + x) = \frac{\partial F}{\partial x} + 1$$
<br>
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial F}{\partial x} + \frac{\partial L}{\partial y} \cdot 1$$

<br><br>
여기서 $\frac{\partial L}{\partial y} \cdot 1$이 바로 skip 경로를 통해 흐르는 gradient $\frac{\partial L}{\partial x_{skip}}$이며, 결과적으로 $\frac{\partial L}{\partial y}$와 동일해진다

Backpropagation 관점에서 덧셈 노드는 상류에서 전달된 gradient를 하류의 모든 입력 분기로 그대로 복사하여 전달하는 성질을 가진다.
왜냐하면 출력 y가 각 입력(F와 x)의 단순 합으로 정의되기 때문이다. 변수 x가 미세하게 $\Delta x$만큼 변할 때, 결과값 $y$도 동일하게 $\Delta x$만큼 변하므로 변화율(Local Gradient)이 $1$이 되기 때문이다.
<br>
강의자료 8장 26페이지에서 가중치 층(CONV layers)을 통과하는 경로는 복잡한 연산을 거치며 기울기가 작아질 수 있지만, Skip 경로(Identity x)는 출력 $y$와 직접적으로 덧셈으로 연결되어 있다. 이 덕분에 상류의 오차 신호($\frac{\partial L}{\partial y}$)가 flow unchanged로 전달된다

(1) Plain 버전 $y_{plain} = F(x)$
<br> Plain 버전에서는 출력이 오직 가중치 층($F$)을 통해서만 전달

<br> $y = F$ 이므로 $\frac{\partial L}{\partial F} = \frac{\partial L}{\partial y} = [1.06, -0.38]$

<br> $F = W_2 \cdot h_1 + b_2$ 이므로
- $\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial F} \cdot \frac{\partial F}{\partial W_2} = \frac{\partial L}{\partial F} \cdot h_1^T$
- $\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial F} \cdot \frac{\partial F}{\partial b_2} = \frac{\partial L}{\partial F}$

<br> $\frac{\partial L}{\partial h_1} = W_2^T \cdot \frac{\partial L}{\partial F}$

<br>
$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial h_1} \cdot \text{ReLU}'(z_1)$

<br><br>
$z_1 = W_1 \cdot x + b_1$ 이므로
$$\frac{\partial L}{\partial x}_{plain} = W_1^T \cdot \frac{\partial L}{\partial z_1}$$

(2) Residual 버전 $y = F(x) + x$으로 위에서 구한 것처럼 Skip connection 추가된다.
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial F}{\partial x} + \frac{\partial L}{\partial y} \cdot 1 = \frac{\partial L}{\partial x}_{plain} + \frac{\partial L}{\partial x_{skip}}$$



plain 버전은 gradient가 곱셈 연산으로만 전달되기에 층이 깊어질 수록 1이하의 값을 계속 곱하면서 gradient가 소실될 수 있다. 그러나 Residual버전은 gradient가 0이 되어도 뒤의 항 $\frac{\partial L}{\partial y}$이 남기에 소실 문제 해결

#### A2. Parameter & FLOP Countingd

1.학습 가능한 파라미터 수 <br>
(i). Convolution 레이어 (2개):
- 하나의 Conv 레이어 파라미터 = $K \times K \times C_{in} \times C_{out}$ (Bias 없음)
- 계산: $3 \times 3 \times 16 \times 16 = 2,304$
- 2개의 레이어이므로: $2,304 \times 2 = 4,608$개

(ii). Batch Normalization 레이어 (2개):
- BN은 채널당 $\gamma$(Scale)와 $\beta$(Shift) 파라미터를 가지므로
- 하나의 BN 레이어 파라미터 = $2 \times C_{out}$
- 계산: $2 \times 16 = 32$
- 2개의 레이어이므로: $32 \times 2 = 64$개
- 총합: $4,608 + 64 = 4,672$개

---
2.MACs <br>
공식: $K \times K \times C_{in} \times H_{out} \times W_{out} \times C_{out}$ <br>
계산:
- 하나의 Conv 레이어: $3 \times 3 \times 16 \times 32 \times 32 \times 16 = 2,359,296$
- 2개의 레이어이므로: $2,359,296 \times 2 =4,718,592 $ MACs

---
3.Downsampling block <br>
(i) 학습 가능한 파라미터 수
1. Main Path (Conv 2개):
- Conv 1: $3 \times 3 \times 16 \times 32 = 4,608$
- Conv 2: $3 \times 3 \times 32 \times 32 = 9,216$
2. Skip Path (1x1 Conv 1개):
- $1 \times 1 \times 16 \times 32 = 512$
3. BN 레이어 (3개 - Main 2개, Skip 1개):
- 각 BN은 출력 채널 수에 비례: $(2 \times 32) + (2 \times 32) + (2 \times 32) = 192$
4. 총합: $4,608 + 9,216 + 512 + 192 = 14,528$개 <br>

(ii) MACs
1. Conv 1 ($3 \times 3, stride=2$):
- $3 \times 3 \times 16 \times (32/2) \times (32/2) \times 32 = 1,179,648$
2. Conv 2 ($3 \times 3, stride=1$):
- $3 \times 3 \times 32 \times 16 \times 16 \times 32 = 2,359,296$
3. Skip Path ($1 \times 1, stride=2$):
- $1 \times 1 \times 16 \times 16 \times 16 \times 32 = 131,072$
4. 총합: $1,179,648 + 2,359,296 + 131,072 = 3,670,016$ MACs

(iii) 1×1 projection 필요 이유 <br>
이전 블록은 입력과 출력의 크기가 같았지만, 이번 블록은 입력의 채널 수(16)와 크기(W x H, 32 x 32)가 출력(32, 16 x 16)과 다르기 때문에 skip connection에서 덧셈 연산을 수행할 수 있도록 차원(dimension)을 일치시키기 위해 $1 \times 1$ projection이 필요


#### A3. BatchNorm Forward by Hand

입력 $X = \begin{bmatrix} 2.0 & 1.0 \\ 4.0 & 3.0 \\ 0.0 & -1.0 \\ -2.0 & 1.0 \end{bmatrix}$ (4개 샘플, 2개 feature) <br><br>
파라미터: $\gamma = [1.5, 0.5]$, $\beta = [0.0, -1.0]$ <br> <br>
상수: $\epsilon = 1e-5$

----
feature별 평균($\mu$)과 분산($\sigma^2$) 계산 - 각 열(column)에 대해 독립적으로 계산 <br>

Feature 1 (첫 번째 열):<br>
$\mu_1 = \frac{2.0 + 4.0 + 0.0 + (-2.0)}{4} = \frac{4.0}{4} = {1.0}$ <br>
$\sigma^2_1 = \frac{(2-1)^2 + (4-1)^2 + (0-1)^2 + (-2-1)^2}{4} = \frac{1 + 9 + 1 + 9}{4} = \frac{20}{4} = {5.0}$ <br><br>
Feature 2 (두 번째 열): <br>
$\mu_2 = \frac{1.0 + 3.0 + (-1.0) + 1.0}{4} = \frac{4.0}{4} = {1.0}$<br>
$\sigma^2_2 = \frac{(1-1)^2 + (3-1)^2 + (-1-1)^2 + (1-1)^2}{4} = \frac{0 + 4 + 4 + 0}{4} = \frac{8}{4} = {2.0}$


---
공식: $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$ <br>
Feature 1 정규화 ($\sqrt{5 + 1e-5} \approx 2.236$):<br>
- $\hat{x}_{1,1} = (2.0 - 1.0) / 2.236 \approx {0.447}$
- $\hat{x}_{2,1} = (4.0 - 1.0) / 2.236 \approx {1.342}$
- $\hat{x}_{3,1} = (0.0 - 1.0) / 2.236 \approx {-0.447}$
- $\hat{x}_{4,1} = (-2.0 - 1.0) / 2.236 \approx {-1.342}$ <br>
<br> Feature 2 정규화 ($\sqrt{2 + 1e-5} \approx 1.414$):<br>
- $\hat{x}_{1,2} = (1.0 - 1.0) / 1.414 = {0.0}$
- $\hat{x}_{2,2} = (3.0 - 1.0) / 1.414 \approx {1.414}$
- $\hat{x}_{3,2} = (-1.0 - 1.0) / 1.414 \approx {-1.414}$
- $\hat{x}_{4,2} = (1.0 - 1.0) / 1.414 = {0.0}$ <br>

---
공식: $y_i = \gamma \hat{x}_i + \beta$ <br><br>
Feature 1 ($y_1 = 1.5 \hat{x}_1 + 0.0$): <br>
- $[0.671, 2.013, -0.671, -2.013]$ <br>

<br> Feature 2 ($y_2 = 0.5 \hat{x}_2 - 1.0$): <br>
- $y_{1,2} = 0.5(0.0) - 1.0 = {-1.0}$
- $y_{2,2} = 0.5(1.414) - 1.0 \approx {-0.293}$
- $y_{3,2} = 0.5(-1.414) - 1.0 \approx {-1.707}$
- $y_{4,2} = 0.5(0.0) - 1.0 = {-1.0}$ <br>

최종 $y$ 행렬:<br>
$$y = \begin{bmatrix} 0.671 & -1.0 \\ 2.013 & -0.293 \\ -0.671 & -1.707 \\ -2.013 & -1.0 \end{bmatrix}$$


evaluation 시에는 현재 배치의 통계량 대신 학습 단계에서 미리 계산하여 저장해 둔 Running Mean과 Running Variance을 고정하여 사용함으로써 배치 크기에 상관없이 Deterministic인 출력을 보장한다. 이는 개별 샘플을 추론할 때 결과가 배치 구성에 따라 변하는 것을 방지하고 학습 시 파악한 전체 데이터의 분포를 일관되게 반영하기 위해 필수적이다

---
## Part B. Implementing & Training ResNet-20



In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import random
import numpy as np

# 1. 시드 고정 (재현성)
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. 모델 정의 (He et al. Specification)
def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)

class BasicBlock(nn.Module):
    def __init__(self, inplanes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Identity()
        if stride != 1 or inplanes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return self.relu(out)

class ResNet20(nn.Module):
    def __init__(self):
        super(ResNet20, self).__init__()
        self.inplanes = 16
        # Stem: 3x3 conv, 16 channels
        self.conv1 = conv3x3(3, 16)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Three stages, n=3 blocks each
        self.layer1 = self._make_layer(16, 3, stride=1)
        self.layer2 = self._make_layer(32, 3, stride=2)
        self.layer3 = self._make_layer(64, 3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, 10)

    def _make_layer(self, planes, blocks, stride):
        layers = []
        layers.append(BasicBlock(self.inplanes, planes, stride))
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [13]:
# 3. 데이터 로더 (No Augmentation 설정)
def get_b1_dataloaders(batch_size=128):
    stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    # 과제 요구사항: No Augmentation
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    return trainloader, testloader

In [ ]:
# 4. 학습 함수 (40 Epochs, Cosine Annealing)
def train_b1_baseline():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = ResNet20().to(device)

    # 파라미터 수 확인 (약 0.27M 확인용)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameter Count: {total_params / 1e6:.6f}M")

    trainloader, testloader = get_b1_dataloaders()

    # 과제 설정: SGD(momentum 0.9), LR 0.1, Weight Decay 0
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=0)

    # 과제 설정: Cosine Learning Rate Schedule (40 epochs)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

    train_losses, test_losses, test_accs = [], [], []

    for epoch in range(40):
        model.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Test Loss 및 Accuracy 계산
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        avg_train_loss = running_loss / len(trainloader)
        avg_test_loss = test_loss / len(testloader)
        acc = 100. * correct / total

        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        test_accs.append(acc)

        scheduler.step()
        print(f"Epoch [{epoch+1}/40] Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Test Acc: {acc:.2f}%")

    # 결과 리포트 및 그래프
    print(f"\nFinal Test Accuracy: {test_accs[-1]:.2f}%")
    plot_results(train_losses, test_losses)

def plot_results(train_losses, test_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title('B1 Baseline: Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

# 실행
train_b1_baseline()

Total Parameter Count: 0.272474M
Epoch [1/40] Train Loss: 1.5896 | Test Loss: 1.7858 | Test Acc: 39.96%
